In [12]:
import requests

from pyiceberg.catalog import load_catalog

POLARIS_URI = "http://polaris:8181/api/catalog"
POLARIS_WAREHOUSE = "default"  # или "rest", если ты так назвал warehouse
POLARIS_REALM = "default-realm"

POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-root-secret-change-me"
POLARIS_SCOPE = "PRINCIPAL_ROLE:ALL"

S3_ENDPOINT = "http://minio:9000"
S3_ACCESS_KEY_ID = "admin"
S3_SECRET_ACCESS_KEY = "password"
S3_REGION = "us-east-1"


token_response = requests.post(
    f"{POLARIS_URI}/v1/oauth/tokens",
    headers={
        "Content-Type": "application/x-www-form-urlencoded",
        "Polaris-Realm": POLARIS_REALM,
    },
    data={
        "grant_type": "client_credentials",
        "client_id": POLARIS_CLIENT_ID,
        "client_secret": POLARIS_CLIENT_SECRET,
        "scope": POLARIS_SCOPE,
    },
)

token_response.raise_for_status()
token = token_response.json()["access_token"]

catalog = load_catalog(
    "polaris",
    **{
        "type": "rest",
        "uri": POLARIS_URI,
        "warehouse": POLARIS_WAREHOUSE,

        # Передаём уже готовый token
        "token": token,

        # Realm для Polaris
        "header.Polaris-Realm": POLARIS_REALM,

        # Важно: отключаем vended credentials / read delegation
        "rest.vended-credentials-enabled": "false",
        "vended-credentials-enabled": "false",

        # FileIO
        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",

        # MinIO credentials напрямую
        "s3.endpoint": S3_ENDPOINT,
        "s3.region": S3_REGION,
        "s3.access-key-id": S3_ACCESS_KEY_ID,
        "s3.secret-access-key": S3_SECRET_ACCESS_KEY,

        # Важно для MinIO
        "s3.force-virtual-addressing": "false",
    },
)

In [4]:
print(token_response.status_code)
print(token_response.json())

200
{'access_token': 'principal:root;password:polaris-root-secret-change-me;realm:default-realm;role:ALL', 'scope': 'PRINCIPAL_ROLE:ALL', 'token_type': 'bearer', 'expires_in': 3600}


In [13]:
print("Namespaces:")
print(catalog.list_namespaces())

print("Tables in my:")
print(catalog.list_tables("my"))

Namespaces:
[('my',)]
Tables in my:
[('my', 'events'), ('my', 't1')]


In [14]:
table = catalog.load_table("my.events")

print(table)
print(table.schema())
print(table.metadata_location)

events(
  1: event_id: optional long,
  2: user_id: optional string,
  3: event_type: optional string,
  4: amount: optional double,
  5: created_at: optional timestamp
),
partition by: [],
sort order: [],
snapshot: Operation.APPEND: id=2118421036277119755, parent_id=3785823644131661367, schema_id=0
table {
  1: event_id: optional long
  2: user_id: optional string
  3: event_type: optional string
  4: amount: optional double
  5: created_at: optional timestamp
}
s3://rest-lakehouse/my/events-6f3484ebf45946adb2f82e1059ac0d18/metadata/00002-c2fe6253-7531-4ec8-992e-41f0c3944e0b.metadata.json


In [15]:
arrow_table = table.scan().to_arrow()

print(arrow_table)

pyarrow.Table
event_id: int64
user_id: string
event_type: string
amount: double
created_at: timestamp[us]
----
event_id: [[1,2,3,4,5,...,9996,9997,9998,9999,10000]]
user_id: [["user_00002","user_00003","user_00004","user_00005","user_00006",...,"user_09997","user_09998","user_09999","user_10000","user_00001"]]
event_type: [["click","add_to_cart","purchase","refund","signup",...,"refund","signup","login","logout","page_view"]]
amount: [[25.85,45.45,264.01,-22.11,36.24,...,-54.5,1.93,24.81,49.96,34.26]]
created_at: [[2026-05-14 11:34:58.545092,2026-05-14 11:34:57.545092,2026-05-14 11:34:56.545092,2026-05-14 11:34:55.545092,2026-05-14 11:34:54.545092,...,2026-05-14 08:48:23.545092,2026-05-14 08:48:22.545092,2026-05-14 08:48:21.545092,2026-05-14 08:48:20.545092,2026-05-14 08:48:19.545092]]


In [16]:
arrow_table.to_pandas()

,event_id,user_id,event_type,amount,created_at
0,1,user_00002,click,25.85,2026-05-14 11:34:58.545092
1,2,user_00003,add_to_cart,45.45,2026-05-14 11:34:57.545092
2,3,user_00004,purchase,264.01,2026-05-14 11:34:56.545092
3,4,user_00005,refund,-22.11,2026-05-14 11:34:55.545092
4,5,user_00006,signup,36.24,2026-05-14 11:34:54.545092
...,...,...,...,...,...
9995,9996,user_09997,refund,-54.50,2026-05-14 08:48:23.545092
9996,9997,user_09998,signup,1.93,2026-05-14 08:48:22.545092
9997,9998,user_09999,login,24.81,2026-05-14 08:48:21.545092
9998,9999,user_10000,logout,49.96,2026-05-14 08:48:20.545092


In [17]:
from pyiceberg.expressions import EqualTo

df_user_001 = (
    table
    .scan(row_filter=EqualTo("user_id", "user_09998"))
    .to_pandas()
)

print(df_user_001)

   event_id     user_id event_type  amount                 created_at
0      9997  user_09998     signup    1.93 2026-05-14 08:48:22.545092


In [18]:
from pyiceberg.expressions import EqualTo

df_clicks = (
    table
    .scan(row_filter=EqualTo("event_type", "click"))
    .to_pandas()
)

df_clicks

,event_id,user_id,event_type,amount,created_at
0,1,user_00002,click,25.85,2026-05-14 11:34:58.545092
1,9,user_00010,click,24.41,2026-05-14 11:34:50.545092
2,17,user_00018,click,21.81,2026-05-14 11:34:42.545092
3,25,user_00026,click,9.81,2026-05-14 11:34:34.545092
4,33,user_00034,click,5.56,2026-05-14 11:34:26.545092
...,...,...,...,...,...
1245,9961,user_09962,click,10.83,2026-05-14 08:48:58.545092
1246,9969,user_09970,click,17.83,2026-05-14 08:48:50.545092
1247,9977,user_09978,click,16.37,2026-05-14 08:48:42.545092
1248,9985,user_09986,click,45.88,2026-05-14 08:48:34.545092


# Партиционирование

In [20]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField,
    LongType,
    StringType,
    DoubleType,
    TimestampType,
)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform, BucketTransform

In [21]:
schema = Schema(
    NestedField(
        field_id=1,
        name="event_id",
        field_type=LongType(),
        required=False,
    ),
    NestedField(
        field_id=2,
        name="user_id",
        field_type=StringType(),
        required=False,
    ),
    NestedField(
        field_id=3,
        name="event_type",
        field_type=StringType(),
        required=False,
    ),
    NestedField(
        field_id=4,
        name="amount",
        field_type=DoubleType(),
        required=False,
    ),
    NestedField(
        field_id=5,
        name="created_at",
        field_type=TimestampType(),
        required=False,
    ),
)

In [22]:
partition_spec = PartitionSpec(
    PartitionField(
        source_id=5,          # created_at
        field_id=1000,
        transform=DayTransform(),
        name="created_at_day",
    ),
    PartitionField(
        source_id=2,          # user_id
        field_id=1001,
        transform=BucketTransform(32),
        name="user_id_bucket",
    ),
)

In [23]:
table_identifier = "my.events_py"

table_location = "s3://rest-lakehouse/my/events_py"

In [27]:
table = catalog.create_table(
    identifier=table_identifier,
    schema=schema,
    partition_spec=partition_spec,
    # location=table_location,
    properties={
        "format-version": "2",
        "write.format.default": "parquet",
    },
)

print(table)

events_py(
  1: event_id: optional long,
  2: user_id: optional string,
  3: event_type: optional string,
  4: amount: optional double,
  5: created_at: optional timestamp
),
partition by: [created_at_day, user_id_bucket],
sort order: [],
snapshot: null
